# 1.07 Softmax 回归与分类

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.07_softmax_classification.ipynb)


## 为什么要学

分类问题无处不在：

- **语言模型**每一步都在做一个"25 万类"的分类：下一个 token 是哪个？
- **RT-2、OpenVLA** 把连续动作切成 256 个"格子"（bin），把动作预测变成分类问题；π0-FAST 用 DCT + BPE 把动作变成 token；
- **离散动作的 RL 策略**（如 DQN、CartPole 上的策略梯度）输出每个动作的概率。

本课掌握三个核心：**logits → softmax → 交叉熵**，以及数值稳定的写法。参考：[d2l 3.4 softmax 回归](https://zh.d2l.ai/chapter_linear-networks/softmax-regression.html)。

---

## 1. 从 logits 到概率：softmax

模型对 $K$ 个类别输出 $K$ 个任意实数 $\mathbf{z} = [z_1,\dots,z_K]$，称为 **logits**。softmax 把它们变成概率：

$$
\text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}
$$

性质：每个值在 $(0,1)$ 之间、和为 1；保持大小顺序；**所有 logits 同时加一个常数，结果不变**（这一点是数值稳定的关键）。

### 数值稳定性

$e^{1000}$ 会溢出成 `inf`。利用"平移不变性"，先减去最大值：

$$
\text{softmax}(\mathbf{z}) = \text{softmax}(\mathbf{z} - \max_j z_j)
$$

同理，$\log\sum_j e^{z_j}$（log-sum-exp）也这样算。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

z = torch.tensor([1000.0, 1001.0, 1002.0])
naive = z.exp() / z.exp().sum()
print("朴素写法:", naive)                         # nan：inf / inf
print("PyTorch :", torch.softmax(z, dim=0))       # 内部做了减最大值

### 练习 1：数值稳定的 softmax 与 log_softmax

In [ ]:
def stable_softmax(z, dim=-1):
    # TODO: 减去 dim 上的最大值（keepdim=True）后再 exp、归一化
    raise NotImplementedError

def stable_log_softmax(z, dim=-1):
    # TODO: log_softmax(z) = z - logsumexp(z) = (z - m) - log(sum(exp(z - m)))
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def stable_softmax(z, dim=-1):
    m = z.max(dim=dim, keepdim=True).values
    e = (z - m).exp()
    return e / e.sum(dim=dim, keepdim=True)

def stable_log_softmax(z, dim=-1):
    m = z.max(dim=dim, keepdim=True).values
    return (z - m) - (z - m).exp().sum(dim=dim, keepdim=True).log()

In [ ]:
# ✅ 自测
zz = torch.randn(4, 10) * 50 + 500
assert torch.allclose(stable_softmax(zz), torch.softmax(zz, -1), atol=1e-6)
assert torch.allclose(stable_log_softmax(zz), torch.log_softmax(zz, -1), atol=1e-4)
print("通过 ✅")

---

## 2. 交叉熵损失

真实类别为 $y$ 时，损失是"模型给正确类别的概率"的负对数：

$$
\ell(\mathbf{z}, y) = -\log\text{softmax}(\mathbf{z})_y = -z_y + \log\sum_j e^{z_j}
$$

它的梯度极其简洁：$\frac{\partial\ell}{\partial z_k} = \text{softmax}(\mathbf{z})_k - \mathbb{1}[k=y]$，即 **"预测概率 − 真实 one-hot"**。

> PyTorch 的 `F.cross_entropy(logits, target)` 直接接收 **logits**（不要先 softmax！），内部用 log-sum-exp 保证稳定。这是新手最常犯的错误之一。

### 练习 2：手写交叉熵

In [ ]:
def my_cross_entropy(logits, target):
    """logits: (B, K)，target: (B,) 整数类别 -> 标量（batch 平均）"""
    # TODO: 用 stable_log_softmax，再用高级索引取出正确类别的 log 概率
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def my_cross_entropy(logits, target):
    """logits: (B, K)，target: (B,) 整数类别 -> 标量（batch 平均）"""
    logp = stable_log_softmax(logits, dim=-1)              # (B, K)
    return -logp[torch.arange(len(target)), target].mean()

In [ ]:
# ✅ 自测
lg = torch.randn(16, 5); tg = torch.randint(0, 5, (16,))
assert torch.allclose(my_cross_entropy(lg, tg), F.cross_entropy(lg, tg), atol=1e-6)
print("通过 ✅")

---

## 3. 训练一个 softmax 分类器

生成三类二维数据（三条螺旋线），先用**线性**分类器，看看它的局限性。

In [ ]:
def make_spirals(n_per_class=200, K=3, noise=0.15, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(K):
        r = np.linspace(0.05, 1, n_per_class)
        t = np.linspace(k * 4, (k + 1) * 4, n_per_class) + rng.normal(0, noise, n_per_class)
        X.append(np.stack([r * np.sin(t), r * np.cos(t)], 1)); y.append(np.full(n_per_class, k))
    return torch.tensor(np.concatenate(X), dtype=torch.float32), torch.tensor(np.concatenate(y))

X, y = make_spirals()

def train_classifier(model, X, y, steps=2000, lr=0.05):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        loss = F.cross_entropy(model(X), y)
        opt.zero_grad(); loss.backward(); opt.step()
    acc = (model(X).argmax(-1) == y).float().mean().item()
    return loss.item(), acc

def plot_decision(model, X, y, title):
    xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 200), np.linspace(-1.2, 1.2, 200))
    grid = torch.tensor(np.stack([xx.ravel(), yy.ravel()], 1), dtype=torch.float32)
    with torch.no_grad():
        zz = model(grid).argmax(-1).reshape(xx.shape)
    plt.contourf(xx, yy, zz, alpha=0.3, cmap="brg"); plt.scatter(X[:, 0], X[:, 1], c=y, s=5, cmap="brg"); plt.title(title)

torch.manual_seed(0)
linear = nn.Linear(2, 3)
loss, acc = train_classifier(linear, X, y)
plt.figure(figsize=(4, 4)); plot_decision(linear, X, y, f"linear softmax  acc={acc:.2f}"); plt.show()

线性分类器只能画**直线**边界，准确率只有 50% 左右。下一课加入**隐藏层和非线性激活**，就能把螺旋分开。

---

## 4. 连续动作离散化：RT-2 / OpenVLA 的做法

把每一维连续动作 $a\in[-1,1]$ 均匀切成 $K$ 个格子，每个格子是一个"类别"（一个 token）。模型用交叉熵学习分类，推理时取格子中心作为动作。

$$
\text{bin}(a) = \min\left(\left\lfloor\frac{a+1}{2}K\right\rfloor, K-1\right),\qquad
\text{center}(k) = \frac{2(k+0.5)}{K} - 1
$$

### 练习 3：动作离散化与还原

In [ ]:
def discretize(a, K=256):
    """a: 任意形状, 取值 [-1, 1] -> 同形状 long 张量, 取值 0..K-1"""
    # TODO: 记得先 clamp 到 [-1, 1]
    raise NotImplementedError

def undiscretize(k, K=256):
    """k: long 张量 -> 格子中心的连续值"""
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def discretize(a, K=256):
    a = a.clamp(-1, 1)
    return ((a + 1) / 2 * K).floor().long().clamp(max=K - 1)

def undiscretize(k, K=256):
    return 2 * (k.float() + 0.5) / K - 1

In [ ]:
# ✅ 自测
a = torch.rand(1000, 7) * 2 - 1
k = discretize(a)
assert k.min() >= 0 and k.max() <= 255 and k.dtype == torch.long
err = (undiscretize(k) - a).abs().max().item()
print(f"量化误差最大 {err:.5f}（理论上限 1/K = {1/256:.5f}）")
assert err <= 1 / 256 + 1e-6
print("通过 ✅")

> **思考**：256 个格子在 [-1,1] 上的分辨率约 0.008。如果一个关节的动作范围是 ±3 rad，量化误差约 0.012 rad ≈ 0.7°。对"线头对插"这种毫米级任务够吗？这就是 π0 选择**流匹配输出连续动作**、而不是离散 token 的动机之一。

---

## 小结

- logits → softmax → 交叉熵；`F.cross_entropy` 直接吃 logits。
- 数值稳定：先减最大值（log-sum-exp 技巧）。
- 交叉熵梯度 = 预测概率 − one-hot。
- 线性模型的决策边界是直线，复杂问题需要非线性（下一课）。
- 连续动作可以离散化成 token（RT-2/OpenVLA），代价是量化误差。